## DATABRICKS CAPSTONE PROJECTS
<br>**DATE: 2026-09-02**
<br>**TOPIC: Understand the loaded data**

In [0]:
from pyspark.sql.functions import *
df = spark.sql("""SHOW TABLES IN ecommerce.bronze""").select(concat_ws('.',col('database'), col('tableName')).alias('table_name'))


df_customer = spark.table('ecommerce.bronze.customers')
df_customer.count()
# 5000

df_customer = df_customer.dropDuplicates()

# get list of all string columns--REMOVE WHITE SPACE
string_cols = [k for k,v in df_customer.dtypes if (v) == 'string']

for i in string_cols:
    df_customer = df_customer.withColumn(i,trim(i))

df_customer.display()

## convert email to lower case

df_customer = df_customer.withColumn('email', lower(col('email')))
df_customer.display()

# 10000

## Handle missing emails/phone number
cols = ['email','phone']
for i in cols:
    df_customer.filter(col(i)=='NA').display()

cols = ['email','phone']
for i in cols:
    df_customer = df_customer.withColumn(i, when(col(i).isNull(), lit('NA')).otherwise(col(i)))


## Standardise City/state name

cols = ['city','state','customer_status']
for i in cols:
    df_customer = df_customer.withColumn(i,initcap(col(i)))


# VALIDATE CUSTOMER STATUS
df_customer.select(col('customer_status')).distinct().display()

valid_status = ['Active','Inactive']
df_customer = df_customer.withColumn('customer_status',when(col('customer_status').isin(valid_status),col('customer_status')).otherwise(None))

#